# Attention Vector Extraction — Colab

Extracts Q, K, V attention vectors from **Llama 3.1 8B**.

**How it works:**
- Extracts to Colab's **local SSD** (no Google Drive needed)
- After each task finishes: **tars it in background** (no compression — just bundling, very fast)
- All tars land in `/content/tars/` — download them at the end or after each phase
- If the runtime dies, re-run: already-downloaded tasks are safe on your machine

**Requirements:**
- **A100 GPU** (40GB+ VRAM). T4 cannot handle 128K context.
- ~50 GB local disk (Colab provides ~100-350 GB)

## 1. Check GPU

In [ ]:
import torch, shutil, os

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU. Go to Runtime > Change runtime type > A100."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
disk_free_gb = shutil.disk_usage("/content").free / 1e9

print(f"GPU:  {gpu_name} ({gpu_mem_gb:.0f} GB VRAM)")
print(f"Disk: {disk_free_gb:.0f} GB free")

if gpu_mem_gb < 30:
    print(
        f"\nWARNING: {gpu_mem_gb:.0f} GB VRAM may not fit 128K context.\n"
        f"Consider A100, or reduce max_length in the config."
    )

## 2. Clone repo, install deps, HF login

In [ ]:
# ── EDIT THIS ──
REPO_URL = "https://github.com/YuvalShemla/LoSeM-attention.git"
BRANCH = "main"

REPO_DIR = "/content/LoSeM-attention"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
!pip install -q transformers accelerate datasets pyyaml tqdm

## 3. Setup paths and config

In [ ]:
import yaml, json, time, subprocess, threading
from pathlib import Path

# Local SSD paths
DATA_ROOT = Path("/content/extraction_data")
TAR_DIR = Path("/content/tars")
DATA_ROOT.mkdir(exist_ok=True)
TAR_DIR.mkdir(exist_ok=True)

# Load config
CONFIG_PATH = Path(REPO_DIR) / "src" / "extraction" / "extraction_config.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

p2_heads = config["phase2"].get("heads", "from_stats")
if p2_heads == "from_stats":
    p2_desc = (f"from_stats, top "
               f"{config['phase2']['n_heads_to_select']}")
elif p2_heads == "all":
    p2_desc = "all heads"
else:
    p2_desc = f"{len(p2_heads)} explicit pairs"

print("Config:")
print(f"  Model:      {config['model']['hf_name']}")
print(f"  Tasks:      {config['tasks']}")
print(f"  Layers:     {config['extraction']['layers']}")
print(f"  Max length: {config['extraction']['max_length']:,}")
print(f"  Phase 1:    {config['phase1']['examples_per_task']} ex/task, all heads")
print(f"  Phase 2:    {config['phase2']['examples_per_task']} ex/task, "
      f"heads: {p2_desc}")

In [ ]:
# ── Optional: override config ──
# Uncomment lines to change, then run this cell.

OVERRIDES = {
    # "extraction": {"max_length": 32768},
    # "extraction": {"layers": [0, 31]},
    # "tasks": ["math_calc", "passkey"],
    # "phase2": {"examples_per_task": 5},
}

if OVERRIDES:
    for section, values in OVERRIDES.items():
        if section in config and isinstance(config[section], dict):
            config[section].update(values)
        else:
            config[section] = values
    with open(CONFIG_PATH, "w") as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)
    print("Config updated.")
else:
    print("No overrides.")

## 4. Background tar helper

After each task finishes extraction, we tar it in a background thread (no compression —
bfloat16 data is incompressible, so tar just bundles files at I/O speed).

In [ ]:
tar_threads = []

def tar_in_background(src_dir, tar_path, delete_after=False):
    """Tar a directory in a background thread (no compression)."""
    def _do_tar():
        t0 = time.time()
        src = Path(src_dir)
        subprocess.run(
            ["tar", "cf", str(tar_path), src.name],
            cwd=str(src.parent), check=True,
        )
        size_mb = Path(tar_path).stat().st_size / 1e6
        print(f"  [tar] {tar_path.name} ready ({size_mb:.0f} MB, {time.time()-t0:.0f}s)")
        if delete_after:
            shutil.rmtree(src_dir, ignore_errors=True)
            print(f"  [tar] deleted {src_dir} to free disk")

    t = threading.Thread(target=_do_tar, daemon=True)
    t.start()
    tar_threads.append(t)
    return t

def wait_for_tars():
    """Wait for all background tars to finish."""
    for t in tar_threads:
        t.join()
    print(f"All {len(tar_threads)} tars complete.")

def list_tars():
    """Show all ready tars with sizes."""
    tars = sorted(TAR_DIR.glob("*.tar"))
    total = 0
    for t in tars:
        sz = t.stat().st_size
        total += sz
        print(f"  {t.name:40s}  {sz/1e6:8.0f} MB")
    print(f"  {'TOTAL':40s}  {total/1e6:8.0f} MB")
    return tars

## 5. Run Phase 1 — task by task

Extracts all heads for 1 example per task. After each task finishes:
- Tars that task's vectors in the background
- Starts extracting the next task immediately

Head statistics (tiny JSON files) are tarred once at the end.

In [ ]:
os.chdir(REPO_DIR)

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "No HF_TOKEN in Colab secrets — add it in the key icon on the left sidebar"

tasks = config["tasks"]
out_cfg = config.get("output", {})
vectors_sub = out_cfg.get("vectors_subdir", "vectors/llama3.1_8b")

print(f"Phase 1: {len(tasks)} tasks, extracting one at a time\n")

for i, task in enumerate(tasks):
    print(f"{'='*50}")
    print(f"  [{i+1}/{len(tasks)}] {task}")
    print(f"{'='*50}")

    task_config = dict(config)
    task_config["tasks"] = [task]
    tmp_cfg = f"/content/_cfg_{task}.yaml"
    with open(tmp_cfg, "w") as f:
        yaml.dump(task_config, f, default_flow_style=False, sort_keys=False)

    data_root = str(DATA_ROOT)
    token = HF_TOKEN

    !python -m src.extraction.extract_vectors \
        --phase 1 \
        --config {tmp_cfg} \
        --data-root {data_root} \
        --hf-token {token}

    os.remove(tmp_cfg) if os.path.exists(tmp_cfg) else None

    task_vec_dir = DATA_ROOT / vectors_sub / "all_heads" / task
    if task_vec_dir.exists():
        tar_path = TAR_DIR / f"phase1_all_heads_{task}.tar"
        tar_in_background(task_vec_dir, tar_path, delete_after=True)
    else:
        print(f"  WARNING: {task_vec_dir} not found, skipping tar")

stats_dir = DATA_ROOT / out_cfg.get("head_stats_subdir", "head_statistics/llama3.1_8b")
if stats_dir.exists():
    tar_in_background(stats_dir, TAR_DIR / "head_statistics.tar")

bench_dir = DATA_ROOT / out_cfg.get("benchmarks_subdir", "benchmarks")
if bench_dir.exists():
    tar_in_background(bench_dir, TAR_DIR / "benchmarks.tar")

wait_for_tars()
print("\nPhase 1 tars:")
list_tars();

## 6. Download Phase 1 tars

Run this cell to download all Phase 1 tars to your local machine.

In [ ]:
from google.colab import files

phase1_tars = sorted(TAR_DIR.glob("phase1_*.tar")) + \
              sorted(TAR_DIR.glob("head_statistics.tar")) + \
              sorted(TAR_DIR.glob("benchmarks.tar"))

print(f"Downloading {len(phase1_tars)} files...\n")
for t in phase1_tars:
    sz_mb = t.stat().st_size / 1e6
    print(f"  {t.name} ({sz_mb:.0f} MB)")
    files.download(str(t))

## 7. Run Phase 2 — task by task

Same pattern: extract one task, tar in background, move to next.
Phase 2 needs head statistics from Phase 1 (still on disk).

In [ ]:
os.chdir(REPO_DIR)
tar_threads.clear()

print(f"Phase 2: {len(tasks)} tasks, extracting one at a time\n")

for i, task in enumerate(tasks):
    print(f"{'='*50}")
    print(f"  [{i+1}/{len(tasks)}] {task}")
    print(f"{'='*50}")

    task_config = dict(config)
    task_config["tasks"] = [task]
    tmp_cfg = f"/content/_cfg_{task}.yaml"
    with open(tmp_cfg, "w") as f:
        yaml.dump(task_config, f, default_flow_style=False, sort_keys=False)

    data_root = str(DATA_ROOT)
    token = HF_TOKEN

    !python -m src.extraction.extract_vectors \
        --phase 2 \
        --config {tmp_cfg} \
        --data-root {data_root} \
        --hf-token {token}

    os.remove(tmp_cfg) if os.path.exists(tmp_cfg) else None

    task_vec_dir = DATA_ROOT / vectors_sub / "selected_heads" / task
    if task_vec_dir.exists():
        tar_path = TAR_DIR / f"phase2_selected_heads_{task}.tar"
        tar_in_background(task_vec_dir, tar_path, delete_after=True)
    else:
        print(f"  WARNING: {task_vec_dir} not found")

wait_for_tars()
print("\nPhase 2 tars:")
list_tars();

## 8. Download Phase 2 tars

In [ ]:
phase2_tars = sorted(TAR_DIR.glob("phase2_*.tar"))

print(f"Downloading {len(phase2_tars)} files...\n")
for t in phase2_tars:
    sz_mb = t.stat().st_size / 1e6
    print(f"  {t.name} ({sz_mb:.0f} MB)")
    files.download(str(t))

## 9. Reassemble locally

After downloading all tars, reassemble the directory structure on your machine:

```bash
mkdir -p data/vectors/llama3.1_8b/all_heads
mkdir -p data/vectors/llama3.1_8b/selected_heads
mkdir -p data/head_statistics
mkdir -p data/benchmarks

# Phase 1
for f in phase1_all_heads_*.tar; do
    tar xf "$f" -C data/vectors/llama3.1_8b/all_heads/
done

# Phase 2
for f in phase2_selected_heads_*.tar; do
    tar xf "$f" -C data/vectors/llama3.1_8b/selected_heads/
done

# Head stats and benchmarks
tar xf head_statistics.tar -C data/
tar xf benchmarks.tar -C data/
```

Final structure:
```
data/
├── vectors/llama3.1_8b/
│   ├── all_heads/{task}/ex_000/layer_XX.pt
│   └── selected_heads/{task}/ex_000/layer_XX.pt
├── head_statistics/llama3.1_8b/{task}.json
└── benchmarks/{task}.json
```

## Notes

- **Disk management:** Each task's vectors are deleted after tarring (`delete_after=True`).
  The local SSD only needs space for ~1 task of data + all tars at once.

- **Download:** The download cells use `files.download()` which triggers browser save prompts.
  You can also download from the Colab file browser (folder icon on left → `/content/tars/`).

- **Colab timeouts:** Free Colab disconnects after ~90 min idle, Pro after ~24h.
  Keep the tab active. If the runtime dies mid-extraction, re-run from the task
  that was interrupted (edit `tasks` in the config override cell to skip completed tasks).

- **Resuming after disconnect:** Already-downloaded tars are safe on your machine.
  Only the current task's data is lost. Re-run with only the remaining tasks.

- **Smaller test runs:** Override `phase1.examples_per_task: 1`, `phase2.examples_per_task: 2`,
  and use 1-2 tasks to verify the pipeline quickly before doing the full extraction.